# Underfitting, overfitting, and finding the capacity you need

The book's prescription is deliberately counterintuitive: make the model overfit first, then fight it. This notebook runs three capacities and shows why.

**Runs on:** CPU — about 4 minutes &nbsp;·&nbsp; **Slides:** [Chapter 5 — Fundamentals of Machine Learning](../../../course-web-slides/ch05/index.html) &nbsp;·&nbsp; **Section:** 04 — Improving model fit

---

## Three models of different sizes

In [ ]:
import numpy as np
import keras
from keras import layers
from keras.datasets import imdb

(train_data, train_labels), _ = imdb.load_data(num_words=10000)

def vectorize(seqs, dim=10000):
    out = np.zeros((len(seqs), dim), dtype="float32")
    for i, s in enumerate(seqs):
        out[i, s] = 1.
    return out

x = vectorize(train_data)
y = np.asarray(train_labels).astype("float32")

def run(units, epochs=20, name=""):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([layers.Dense(units, activation="relu"),
                          layers.Dense(units, activation="relu"),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer="rmsprop", loss="binary_crossentropy",
              metrics=["accuracy"])
    h = m.fit(x, y, epochs=epochs, batch_size=512,
              validation_split=0.4, verbose=0)
    print(f"{name:12s} best val loss {min(h.history['val_loss']):.4f} "
          f"at epoch {int(np.argmin(h.history['val_loss']))+1}")
    return h

h_small = run(4, name="tiny (4)")
h_medium = run(16, name="medium (16)")
h_large = run(512, name="large (512)")

## The three curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4.8))
for h, lab, c in [(h_small, "tiny (4)", "#1f77b4"),
                  (h_medium, "medium (16)", "#2ca02c"),
                  (h_large, "large (512)", "#d62728")]:
    plt.plot(h.history["val_loss"], c=c, lw=1.6, label=f"{lab} — validation")
    plt.plot(h.history["loss"], c=c, lw=1.0, ls="--", alpha=.6,
             label=f"{lab} — training")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.ylim(0, 0.8)
plt.legend(fontsize=9); plt.title("Capacity, and when overfitting starts")
plt.show()

Read it in three parts.

**Tiny** — training loss stays high. It cannot represent the problem; this is underfitting, and no amount of regularization helps.

**Large** — training loss collapses almost immediately and validation loss turns around within two epochs. Enormous capacity, memorized fast.

**Medium** — the useful one, and note that it *also* overfits. The goal was never a model that does not overfit.

## Why you should make it overfit first

The book's sequence is: **get to overfitting, then regularize**. The reason is that a model that has not overfit tells you nothing — you cannot distinguish *not enough capacity* from *not enough training* from *the wrong architecture* by looking at a flat curve.

Once it overfits, you know the capacity is sufficient and every subsequent change is measurable.

## Three things to check before blaming capacity

In [ ]:
# 1. Is the learning rate right? Chapter 3's lesson, on a real model.
for lr in [1e-5, 1e-3, 1e-1]:
    keras.utils.set_random_seed(0)
    m = keras.Sequential([layers.Dense(16, activation="relu"),
                          layers.Dense(16, activation="relu"),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.RMSprop(learning_rate=lr),
              loss="binary_crossentropy", metrics=["accuracy"])
    h = m.fit(x, y, epochs=5, batch_size=512, validation_split=0.4, verbose=0)
    print(f"lr={lr:<7} train loss after 5 epochs: "
          f"{h.history['loss'][-1]:.4f}   val acc {h.history['val_accuracy'][-1]:.4f}")

Expected output:

```
lr=1e-05   train loss after 5 epochs: 0.6xxx   val acc 0.6xxx
lr=0.001   train loss after 5 epochs: 0.1xxx   val acc 0.88xx
lr=0.1     train loss after 5 epochs: 0.6xxx   val acc 0.5xxx
```

Both extremes look like *the model cannot learn*. Only one of them is about the model. **Check the learning rate, the batch size, and whether the problem is learnable at all** before reaching for a bigger network.

## A problem with no signal in it

In [ ]:
rng = np.random.default_rng(0)
x_junk = rng.random((5000, 100)).astype("float32")
y_junk = rng.integers(0, 2, size=5000).astype("float32")

keras.utils.set_random_seed(0)
m = keras.Sequential([layers.Dense(64, activation="relu"),
                      layers.Dense(64, activation="relu"),
                      layers.Dense(1, activation="sigmoid")])
m.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
h = m.fit(x_junk, y_junk, epochs=30, batch_size=128,
          validation_split=0.3, verbose=0)
print(f"training accuracy:   {h.history['accuracy'][-1]:.3f}")
print(f"validation accuracy: {h.history['val_accuracy'][-1]:.3f}  (chance is 0.5)")

Training accuracy climbs; validation sits at chance. **This is what a genuinely unlearnable problem looks like**, and it is worth recognising, because the response is to go back to the data rather than to the architecture.

---

## What to take away

- Underfitting and overfitting look different on the loss curves and have opposite remedies.
- **Make the model overfit first** — a flat curve is uninformative.
- Check the learning rate and the batch size before concluding the model is too small.
- A model that fits training data while validation sits at chance means the problem, not the model, is the issue.